# 01 — Run Tests Locally with Databricks Connect

Run the full test suite from your local machine using **Databricks Connect**.

Your code runs locally, but Spark operations execute on a remote Databricks cluster or serverless compute. This gives you the best of both worlds: **fast local iteration with real Databricks infrastructure.**

## Setup

Make sure you've installed the dependencies:
```bash
python -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
```

Then configure Databricks Connect. You need one of:
- A `~/.databrickscfg` profile with `host`, `token`, and `cluster_id`
- Environment variables: `DATABRICKS_HOST`, `DATABRICKS_TOKEN`, `DATABRICKS_CLUSTER_ID`
- Or use serverless compute (no cluster ID needed)

> **Note:** `databricks-connect` bundles PySpark. If you have standalone `pyspark` installed, uninstall it first to avoid conflicts.

## Step 1: Look at the code we're testing

Let's peek at `baseball_stats.py` — the functions under test.

In [1]:
# Quick look at the functions we wrote
from baseball_stats import batting_average, slugging_percentage, classify_hitter

# Try them out manually first — these are pure Python, no Spark needed
print("Batting average (150 H / 500 AB):", batting_average(150, 500))
print("Slugging (80 1B, 20 2B, 5 3B, 15 HR / 400 AB):", slugging_percentage(80, 20, 5, 15, 400))
print("Classification for .320:", classify_hitter(0.320))
print("Classification for .210:", classify_hitter(0.210))

Batting average (150 H / 500 AB): 0.3
Slugging (80 1B, 20 2B, 5 3B, 15 HR / 400 AB): 0.487
Classification for .320: Elite
Classification for .210: Below Average


## Step 2: Run pure Python tests

These tests validate the math functions — no Spark involved. They run in milliseconds.

In [2]:
import pytest
import sys

# Run ONLY the pure Python tests
# -v = verbose (show each test name and result)
# -p no:cacheprovider = don't write .pytest_cache (cleaner in notebooks)
retcode = pytest.main([
    "tests/test_pure_python.py",
    "-v",
    "-p", "no:cacheprovider"
])
print(f"\nReturn code: {retcode} ({'PASSED' if retcode == 0 else 'FAILED'})")

============================= test session starts ==============================
platform darwin -- Python 3.12.10, pytest-9.0.2, pluggy-1.6.0 -- /Users/alexander.booth/miniconda3/envs/demo-env/bin/python
rootdir: /Users/alexander.booth/Documents/Repos/databricks_demos/unit-testing-pyspark
plugins: anyio-4.9.0, Faker-40.1.2
collecting ... collected 25 items

tests/test_pure_python.py::TestBattingAverage::test_normal_calculation PASSED [  4%]
tests/test_pure_python.py::TestBattingAverage::test_perfect_average PASSED [  8%]
tests/test_pure_python.py::TestBattingAverage::test_zero_at_bats_returns_zero PASSED [ 12%]
tests/test_pure_python.py::TestBattingAverage::test_hitless PASSED       [ 16%]
tests/test_pure_python.py::TestBattingAverage::test_rounding PASSED      [ 20%]
tests/test_pure_python.py::TestSluggingPercentage::test_singles_only PASSED [ 24%]
tests/test_pure_python.py::TestSluggingPercentage::test_all_home_runs PASSED [ 28%]
tests/test_pure_python.py::TestSluggingPercentage::te

## Step 3: Run Spark transformation tests

These tests use **Databricks Connect** — your code runs locally but Spark operations
execute on a remote cluster or serverless compute. The first run takes a few seconds
to establish the connection; after that, the session is reused.

In [3]:
# Run ONLY the Spark tests
retcode = pytest.main([
    "tests/test_spark_transforms.py",
    "-v",
    "-p", "no:cacheprovider"
])
print(f"\nReturn code: {retcode} ({'PASSED' if retcode == 0 else 'FAILED'})")

============================= test session starts ==============================
platform darwin -- Python 3.12.10, pytest-9.0.2, pluggy-1.6.0 -- /Users/alexander.booth/miniconda3/envs/demo-env/bin/python
rootdir: /Users/alexander.booth/Documents/Repos/databricks_demos/unit-testing-pyspark
plugins: anyio-4.9.0, Faker-40.1.2
collecting ... collected 14 items

tests/test_spark_transforms.py::TestAddBattingAverage::test_adds_column PASSED [  7%]
tests/test_spark_transforms.py::TestAddBattingAverage::test_correct_calculation PASSED [ 14%]
tests/test_spark_transforms.py::TestAddBattingAverage::test_preserves_existing_columns PASSED [ 21%]
tests/test_spark_transforms.py::TestAddBattingAverage::test_row_count_unchanged PASSED [ 28%]
tests/test_spark_transforms.py::TestAddSluggingPct::test_adds_column PASSED [ 35%]
tests/test_spark_transforms.py::TestAddSluggingPct::test_ohtani_slugging PASSED [ 42%]
tests/test_spark_transforms.py::TestAddSluggingPct::test_low_at_bats_player PASSED [ 50%]
test

## Step 4: Run the full suite

In practice, you run everything at once. This is what CI/CD would do on every commit.

In [4]:
# Run ALL tests in the tests/ directory
retcode = pytest.main([
    "tests/",
    "-v",
    "-p", "no:cacheprovider"
])
assert retcode == 0, "Some tests failed — check the output above."

============================= test session starts ==============================
platform darwin -- Python 3.12.10, pytest-9.0.2, pluggy-1.6.0 -- /Users/alexander.booth/miniconda3/envs/demo-env/bin/python
rootdir: /Users/alexander.booth/Documents/Repos/databricks_demos/unit-testing-pyspark
plugins: anyio-4.9.0, Faker-40.1.2
collecting ... collected 39 items

tests/test_pure_python.py::TestBattingAverage::test_normal_calculation PASSED [  2%]
tests/test_pure_python.py::TestBattingAverage::test_perfect_average PASSED [  5%]
tests/test_pure_python.py::TestBattingAverage::test_zero_at_bats_returns_zero PASSED [  7%]
tests/test_pure_python.py::TestBattingAverage::test_hitless PASSED       [ 10%]
tests/test_pure_python.py::TestBattingAverage::test_rounding PASSED      [ 12%]
tests/test_pure_python.py::TestSluggingPercentage::test_singles_only PASSED [ 15%]
tests/test_pure_python.py::TestSluggingPercentage::test_all_home_runs PASSED [ 17%]
tests/test_pure_python.py::TestSluggingPercentage::te

## What Just Happened?

1. **pytest discovered** all files matching `test_*.py` in the `tests/` folder
2. **conftest.py** created a `DatabricksSession` (once) via Databricks Connect and sample DataFrames
3. **Pure Python tests** ran instantly — just math, no Spark needed
4. **Spark tests** created DataFrames and ran transformations on your Databricks cluster
5. **All tests** used fake data we control — no production dependencies

You can run this exact same `pytest tests/ -v` command:
- From your terminal (via Databricks Connect)
- In a Databricks notebook (next notebook)
- In GitHub Actions / CI pipeline
- Against any cluster or serverless compute

The tests don't change. Only the Spark backend does.